In [ ]:
import numpy as np
import pandas as pd
import os
import time
from tqdm import tqdm, trange
import sys
import matplotlib.pyplot as plt
import pickle

# variaveis
L = 100 # lado do lattice
n_lagartos = L**2 # lagartos que cabem no lattice
estrategias = ['O', 'Y', 'B'] # estratégias possíveis
index_map = {'O': 0, 'Y': 1, 'B': 2}
n_geracoes = 10
n_pop = 1 # número de populações independentes
prob_mutacao = None # probabilidade de mutação a cada geração
a
tipo = f"O_{minimo_O}-{maximo_O}_incO{inclinacao_O}_B{minimo_B}-{maximo_B}_incB{inclinacao_B}"

output_dir = "C:\\Unicamp\\mestrado\\simulacoes\\RPS-python\\RPS-POO\\outputs\\custo_Y\\" + tipo + "/"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
class Lagarto:
  def __init__(self, i, j, estrategia, fitness, coord_vizinhos_interacao, estrategia_vizinhos_interacao, coord_vizinhanca_extendida, estrategia_vizinhanca_extendida, t, n_vizinhos_interacao, n_vizinhos_aprendizado):
    self.i = i # linha
    self.j = j # coluna
    self.estrategia = estrategia
    self.fitness = 0 # inicia com 0 de fitness
    self.coord_vizinhos_interacao = [] # lista vazia para adicionar as coordenadas dos vizinhos
    self.estrategia_vizinhos_interacao = [] # lista vazia para adicionar as estratégias dos vizinhos
    self.coord_vizinhos_aprendizado = []
    self.estrategia_vizinhos_aprendizado = []
    self.t = 0 # determina a geracao do lagarto
    self.n_vizinhos_interacao = 0 # número de vizinhos que o lagarto efetivamente joga
    self.n_vizinhos_aprendizado = 0 # número de vizinhos que o lagarto olha para aprender

  def calcular_coord_vizinhos(self, L): # obtém as coordenadas dos vizinhos

    lista_vizinhos_interacao = []
    vizinhos_possiveis_interacao = []
    n_v_i = self.n_vizinhos_interacao

    for dx in range(-3, 4): 
        for dy in range(-3, 4):
            if dx == 0 and dy == 0: # ignora ele mesmo
                continue
            ni = (self.i + dx) % L # fronteiras periódicas
            nj = (self.j + dy) % L # fronteiras periódicas
            vizinhos_possiveis_interacao.append(((ni, nj), max(abs(dx), abs(dy)))) # armazena a coordenada dos vizinhos possíveis e a sua distância até o lagarto atual
        # Ordena os vizinhos por distância (mais próximos primeiro)
        vizinhos_possiveis_interacao.sort(key=lambda x: x[1])
        lista_vizinhos_interacao = [coord for coord, _ in vizinhos_possiveis_interacao[:n_v_i]]
    
    self.coord_vizinhos_interacao = lista_vizinhos_interacao # armazena as coordenadas dos vizinhos na variável do lagarto
    
    lista_vizinhos_aprendizado = []
    vizinhos_possiveis_aprendizado = []
    n_v_a = self.n_vizinhos_aprendizado

    for dx in range(-3, 4): 
        for dy in range(-3, 4):
            if dx == 0 and dy == 0: # ignora ele mesmo
                continue
            ni = (self.i + dx) % L # fronteiras periódicas
            nj = (self.j + dy) % L # fronteiras periódicas
            vizinhos_possiveis_aprendizado.append(((ni, nj), max(abs(dx), abs(dy)))) # armazena a coordenada dos vizinhos possíveis e a sua distância até o lagarto atual
        # Ordena os vizinhos por distância (mais próximos primeiro)
        vizinhos_possiveis_aprendizado.sort(key=lambda x: x[1])
        lista_vizinhos_aprendizado = [coord for coord, _ in vizinhos_possiveis_aprendizado[:n_v_a]]
    
    self.coord_vizinhos_aprendizado = lista_vizinhos_aprendizado # armazena as coordenadas dos vizinhos na variável do lagarto

  def obter_estrategia_vizinhos(self, matriz_posicao):
      self.estrategia_vizinhos_interacao = [matriz_posicao[ni, nj] for ni, nj in self.coord_vizinhos_interacao] # dadas as coordenadas, obtém a estratégia do lagarto que ocupa aquela posição
      self.estrategia_vizinhos_aprendizado = [matriz_posicao[ni, nj] for ni, nj in self.coord_vizinhos_aprendizado] # dadas as coordenadas, obtém a estratégia do lagarto que ocupa aquela posição

  def mutacao(self, prob_mutacao): # função de mutação
    if np.random.rand() < prob_mutacao: # sorteia um valor entre 0 e 1, se for menor que a probabilidade de mutação, o lagarto muda de estratégia
        estrategias_possiveis = [e for e in estrategias if e != self.estrategia] # obtém as estratégias possíveis, exceto a atual
        self.estrategia = np.random.choice(estrategias_possiveis) # escolhe uma nova estratégia aleatoriamente para mutar


### PRECISO DECIDIR QUANTOS VIZINHOS!!!
  def adicionar_vizinhos_inicial(self):
      if self.estrategia == 'Y':
          n_vizinhos_interacao = np.random.randint(1, 49)
          self.n_vizinhos_interacao = n_vizinhos_interacao
      elif self.estrategia == 'O':
          n_vizinhos_interacao = 24
          #n_vizinhos_interacao = np.random.randint(1, 49)
          self.n_vizinhos_interacao = n_vizinhos_interacao
      elif self.estrategia == 'B':
          n_vizinhos_interacao = 8
          #n_vizinhos_interacao = np.random.randint(1, 49)
          self.n_vizinhos_interacao = n_vizinhos_interacao
###

def calcular_media_vizinhos(lagartos, estrategias, tipo):
    if tipo == 'interacao':
        medias_interacao = []
        for e in estrategias:
            #viz = [lag.n_vizinhos_realizado for lag in lagartos if lag.estrategia == e]
            viz = [lag.n_vizinhos_interacao for lag in lagartos if lag.estrategia == e]
            medias_interacao.append(np.mean(viz) if len(viz) > 0 else 0)
        return medias_interacao # retorna a média de vizinhos para cada estratégia

    elif tipo == 'aprendizado':
        medias_aprendizado = []
        for e in estrategias:
            viz = [lag.n_vizinhos_aprendizado for lag in lagartos if lag.estrategia == e]
            medias_aprendizado.append(np.mean(viz) if len(viz) > 0 else 0)
        return medias_aprendizado # retorna a média de vizinhos para cada estratégia

In [ ]:
def criar_lagartos(n_lagartos, L, estrategias): # define as posições e estratégias dos lagartos no t = 0
  lista_lagartos = []

  # posições iniciais aleatórias
  all_positions = [(i, j) for i in range(L) for j in range(L)] # forma todas as posições possíveis em um lattice
  unique_positions_indices = np.random.choice(len(all_positions), n_lagartos, replace=False) # determina o índice de onde vai ficar cada posição
  unique_positions = [all_positions[i] for i in unique_positions_indices] # basicamente, ele embaralhou as posições

  for g in range(n_lagartos):
    i, j = unique_positions[g] # posição na matriz
    estrategia = np.random.choice(estrategias) # sorteia a estrategia
    lista_lagartos.append(Lagarto(i, j, estrategia, 0, [], [], [], [], 0, 0, 0)) # cria o lagarto
  return lista_lagartos


def calcular_fitness(lagarto, index_map, matriz_posicao): # função para calcular o fitness do lagarto
    fitness_total = 0 # inicia no 0

    b = 2
    c = 1.5

    matriz_payoff = np.array([[1, b-c, b],
                              [b, 1, b-c],
                              [b-c, b, 1]])
    
    vizinhos_interacao = set(lagarto.coord_vizinhos_interacao)
    for ni, nj in vizinhos_interacao:
        vizinho_estrat = matriz_posicao[ni, nj] # pega a estratégia do vizinho dadas as suas coordenadas
        if vizinho_estrat is not None:
            fitness_total += matriz_payoff[index_map[lagarto.estrategia], index_map[vizinho_estrat]] # calcula o payoff do lagarto contra o vizinho de acordo com a matriz de payoff e soma ao fitness total
    lagarto.fitness = fitness_total
    return fitness_total

calcular_freq = lambda mat: np.array([np.sum(mat == s) / (L ** 2) for s in ['O', 'Y', 'B']]) # calcula a frequência de cada estratégia no lattice na ordem O, Y, B

In [ ]:
def atualizar_lagartos(lagartos): # função que atualiza as estratégias dos lagartos com base no fitness dos vizinhos
    novas_estrategias = {} # Dicionário para armazenar as novas estratégias
    novas_vizinhancas_aprendizado = {} # Dicionário para armazenar as novas vizinhanças de aprendizado
    novas_vizinhancas_interacao = {} # Dicionário para armazenar as novas vizinhanças de interação

    mapa = {(l.i, l.j): l for l in lagartos} # dicionário para acessar lagartos pela posição

    for lagarto in lagartos:
        melhor_estrategia = lagarto.estrategia # inicia com a própria estratégia
        maior_fitness = lagarto.fitness # verifica o fitness do próprio lagarto
        melhor_vizinhanca = lagarto.n_vizinhos
            
        # verifica o fitness dos vizinhos normais
        for (ni, nj) in lagarto.coord_vizinhos_aprendizado:
            vizinho = mapa[(ni, nj)] # usa o dicionário para achar o vizinho
            if vizinho.fitness > maior_fitness: # se o fitness do vizinho for maior que o maior fitness atual
                maior_fitness = vizinho.fitness # atualiza o maior fitness
                melhor_estrategia = vizinho.estrategia # atualiza a melhor estratégia
                melhor_vizinhanca_aprendizado = vizinho.n_vizinhos_aprendizado # atualiza a melhor vizinhança
                melhor_vizinhanca_interacao = vizinho.n_vizinhos_interacao
            if vizinho.fitness == maior_fitness:
                a = np.random.rand()
                if a < 0.5:
                    maior_fitness = vizinho.fitness # atualiza o maior fitness
                    melhor_estrategia = vizinho.estrategia # atualiza a melhor estratégia
                    melhor_vizinhanca_aprendizado = vizinho.n_vizinhos_aprendizado # atualiza a melhor vizinhança
                    melhor_vizinhanca_interacao = vizinho.n_vizinhos_interacao
                else:
                    pass
                # se houver empate de fitness ou for menor, mantém a estratégia atual (não muda)

        novas_estrategias[(lagarto.i, lagarto.j)] = melhor_estrategia # armazena a nova estratégia no dicionário
        novas_vizinhancas_aprendizado[(lagarto.i, lagarto.j)] = melhor_vizinhanca_aprendizado # armazena a nova vizinhança no dicionário
        novas_vizinhancas_interacao[(lagarto.i, lagarto.j)] = melhor_vizinhanca_interacao # armazena a nova vizinhança no dicionário

     # atualiza as estratégias de todos os lagartos simultaneamente
    for lagarto in lagartos:
        lagarto.estrategia = novas_estrategias[(lagarto.i, lagarto.j)]  # atualiza estratégia
        # Garantir tamanhos fixos para O e B; somente Y herda vizinhança adaptativa
        #if lagarto.estrategia == 'O':
            #lagarto.n_vizinhos = 24
        #elif lagarto.estrategia == 'B':
            #lagarto.n_vizinhos = 8
        #else:  # 'Y'
        lagarto.n_vizinhos_aprendizado = novas_vizinhancas_aprendizado[(lagarto.i, lagarto.j)]
        lagarto.n_vizinhos_interacao = novas_vizinhancas_interacao[(lagarto.i, lagarto.j)]
        #lagarto.n_vizinhos = novas_vizinhancas[(lagarto.i, lagarto.j)]
    
    return lagartos

In [ ]:
# iniciando a simulação
def simulacao(n_geracoes, L, n_lagartos, estrategias, index_map, n_pop, prob_mutacao = None, seed = None):
    matriz_frequencias = np.full((n_geracoes + 1, n_pop, len(estrategias)), np.nan, dtype=float) # cria uma matriz para armazenar as frequências em cada instante dos loops
    matriz_n_vizinhos_aprendizado_media = np.full((n_geracoes + 1, n_pop, len(estrategias)), np.nan, dtype=float) # cria uma matriz para armazenar vizinhos
    n_vizinhos_aprendizado_individual = []  # lista para armazenar todos os números de vizinhos
    matriz_n_vizinhos_interacao_media = np.full((n_geracoes + 1, n_pop, len(estrategias)), np.nan, dtype=float) # cria uma matriz para armazenar vizinhos
    n_vizinhos_interacao_individual = []  # lista para armazenar todos os números de vizinhos
    historico_estrategias = [] # lista para armazenar o histórico de estratégias de cada população

    for pop in range(n_pop): # loop para cada população independente
        if seed is not None:
          np.random.seed(seed + pop) # coloca uma semente diferente pra cada pop, garantindo independência e reproducibilidade

        frequencias = [] # vai armazenar as frequências ao longo das gerações para essa população
        matriz_posicao = np.full((L, L), None) # cria uma matriz vazia com None
        historico_estrategias_pop = []
        n_vizinhos_aprendizado_pop = []
        n_vizinhos_interacao_pop = []

        lista_lagartos = criar_lagartos(n_lagartos, L, estrategias) # cria os lagartos
        for lagarto in lista_lagartos:
            lagarto.adicionar_vizinhos_inicial() # adiciona o número de vizinhos iniciais de acordo com a estratégia
            matriz_posicao[lagarto.i, lagarto.j] = str(lagarto.estrategia) # cria a matriz de posições de acordo com os lagartos

        frequencias.append(calcular_freq(matriz_posicao)) # calcula a frequência inicial
        n_vizinhos_aprendizado_pop.append([lagarto.n_vizinhos_aprendizado for lagarto in lista_lagartos])  # geração inicial
        n_vizinhos_interacao_pop.append([lagarto.n_vizinhos_interacao for lagarto in lista_lagartos])  # geração inicial
        historico_estrategias_pop.append([lagarto.estrategia for lagarto in lista_lagartos])  # histórico de estratégias
        matriz_n_vizinhos_aprendizado_media[0, pop, :] = calcular_media_vizinhos(lista_lagartos, estrategias, 'aprendizado')
        matriz_n_vizinhos_interacao_media[0, pop, :] = calcular_media_vizinhos(lista_lagartos, estrategias, 'interacao')

        for t in range(1, n_geracoes + 1): # loop para cada geração dentro da população
          # determinando os vizinhos
          for lagarto in lista_lagartos:
            lagarto.calcular_coord_vizinhos(L) # calcula as coordenadas dos vizinhos
            lagarto.obter_estrategia_vizinhos(matriz_posicao) # obtém as estratégias dos vizinhos

          # calculando o fitness
          for lagarto in lista_lagartos:
            calcular_fitness(lagarto, index_map, matriz_posicao) # calcula o fitness do lagarto de acordo com seus vizinhos e a matriz de fitness

          lista_lagartos = atualizar_lagartos(lista_lagartos) # atualiza as estratégias dos lagartos de acordo com o maior fitness dos vizinhos

          if prob_mutacao is not None:
            for lagarto in lista_lagartos:
              lagarto.mutacao(prob_mutacao) # aplica a mutação

          # atualiza a matriz de posição com as novas estratégias e com as mutações
          matriz_posicao = np.full((L, L), None)
          for lagarto in lista_lagartos:
            matriz_posicao[lagarto.i, lagarto.j] = str(lagarto.estrategia)
          #print(matriz_posicao)

          n_vizinhos_aprendizado_geracao = [lagarto.n_vizinhos_aprendizado for lagarto in lista_lagartos]
          n_vizinhos_aprendizado_pop.append(n_vizinhos_aprendizado_geracao)
          n_vizinhos_interacao_geracao = [lagarto.n_vizinhos_interacao for lagarto in lista_lagartos]
          n_vizinhos_interacao_pop.append(n_vizinhos_interacao_geracao)
          matriz_n_vizinhos_aprendizado_media[t, pop, :] = calcular_media_vizinhos(lista_lagartos, estrategias, 'aprendizado') # calcula a média de vizinhos para cada estratégia e armazena na matriz n_vizinhos
          matriz_n_vizinhos_interacao_media[t, pop, :] = calcular_media_vizinhos(lista_lagartos, estrategias, 'interacao') # calcula a média de vizinhos para cada estratégia e armazena na matriz n_vizinhos
          #print(matriz_n_vizinhos[t, pop, :]) # debug
          historico_estrategias_pop.append([lagarto.estrategia for lagarto in lista_lagartos])  # histórico de estratégias
          frequencias.append(calcular_freq(matriz_posicao)) # calcula a frequência dessa geração e armazena em frequencias

          for lagarto in lista_lagartos:
              lagarto.t += 1 # incrementa a geração do lagarto
          
        frequencias = np.array(frequencias)
        n_vizinhos_aprendizado_individual.append(n_vizinhos_aprendizado_pop)
        n_vizinhos_interacao_individual.append(n_vizinhos_interacao_pop)
        historico_estrategias.append(historico_estrategias_pop)
        for t in range(n_geracoes + 1):
          matriz_frequencias[t, pop, :] = frequencias[t]

          # >>>>> SALVAR OS DADOS DESTA POPULAÇÃO <<<<<
        with open(os.path.join(output_dir, f"pop_{pop}_n_vizinhos_aprendizado_individual.pkl"), "wb") as f:
          pickle.dump(n_vizinhos_aprendizado_pop, f)
        with open(os.path.join(output_dir, f"pop_{pop}_n_vizinhos_interacao_individual.pkl"), "wb") as f:
          pickle.dump(n_vizinhos_interacao_pop, f)
        with open(os.path.join(output_dir, f"pop_{pop}_historico_estrategias.pkl"), "wb") as f:
          pickle.dump(historico_estrategias_pop, f)
        np.save(os.path.join(output_dir, f"pop_{pop}_matriz_frequencias.npy"), frequencias)
        np.save(os.path.join(output_dir, f"pop_{pop}_matriz_n_vizinhos_aprendizado_media.npy"), matriz_n_vizinhos_aprendizado_media[:, pop, :])
        np.save(os.path.join(output_dir, f"pop_{pop}_matriz_n_vizinhos_interacao_media.npy"), matriz_n_vizinhos_interacao_media[:, pop, :])
            # >>>>> FIM DO SALVAMENTO <<<<<

    return matriz_frequencias, matriz_n_vizinhos_aprendizado_media, matriz_n_vizinhos_interacao_media, n_vizinhos_aprendizado_individual, n_vizinhos_interacao_individual, historico_estrategias

freq, n_vizinhos_aprendizado, n_vizinhos_interacao, n_vizinhos_aprendizado_individual, n_vizinhos_interacao_individual, historico_estrategias = simulacao(n_geracoes, L, n_lagartos, estrategias, index_map, n_pop, minimo_O, maximo_O, inclinacao_O, minimo_B, maximo_B, inclinacao_B, prob_mutacao, seed = 10)